In [3]:
%mamba install pandas

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 2.584 seconds
  Name           Version  Build                Channel
--------------------------------------------------------------------
+ pandas         3.1.0    np23py313h124dbc7_0  emscripten-forge-4x
+ python-tzdata  2026.5   pyh5ded981_0         conda-forge
- pip            26.2.1   pyh145f28c_0         conda-forge


In [4]:
%mamba install scikit-learn

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, scikit-learn
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 1.044 seconds
  Name                Version     Build                Channel
----------------------------------------------------------------------------
+ brotli-python       1.2.0       py313h8faef1e_3      emscripten-forge-4x
+ certifi             2026.10.23  pyhd8ed1ab_0         conda-forge
+ charset-normalizer  3.5.1       pyhd8ed1ab_0         conda-forge
+ idna                3.20        pyh5ded981_0         conda-forge
+ joblib              1.6.0       py313h57e9779_0      emscripten-forge-4x
+ narwhals            2.27.1      pyh5ded981_0         conda-forge
+ pysocks             1.7.1       py313h1804a44_3      emscripten-forge-4x
+ requests            2.34.2      pyhcf101f3_0         conda-forge
+ scikit-learn        1.9.0       np23py313hb9816c2_0  emscripten-forge-4x
+ setupto

In [5]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

DATA_PATH = "E_bearing_condition.csv"  
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())


Shape: (600, 4)
Columns: ['vib_rms_mm_s', 'kurtosis', 'fault', 'split']


,vib_rms_mm_s,kurtosis,fault,split
0,1.851,2.677,0,train
1,2.062,2.746,0,train
2,1.518,3.031,0,train
3,1.597,3.281,0,train
4,2.853,2.789,0,train


In [11]:
# E1 - One feature logistic regression

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Extract data
X_train_E1 = train_df[["vib_rms_mm_s"]]
y_train_E = train_df["fault"].astype(int)

X_test_E1 = test_df[["vib_rms_mm_s"]]
y_test_E = test_df["fault"].astype(int)

# Fit model using default settings
model_E1 = LogisticRegression()
model_E1.fit(X_train_E1, y_train_E)

# Predict probabilities and apply threshold 0.5
prob_E1 = model_E1.predict_proba(X_test_E1)[:, 1]
pred_E1 = (prob_E1 >= 0.5).astype(int)

# Always-healthy baseline
always_healthy = np.zeros(len(y_test_E), dtype=int)
baseline_accuracy = accuracy_score(y_test_E, always_healthy) * 100

# Model performance
accuracy = accuracy_score(y_test_E, pred_E1) * 100
precision = precision_score(
    y_test_E, pred_E1, zero_division=0
) * 100
recall = recall_score(
    y_test_E, pred_E1, zero_division=0
) * 100

# Weight and bias
w = model_E1.coef_[0, 0]
b = model_E1.intercept_[0]

# Decision boundary
boundary = -b / w if w != 0 else np.nan

# Print only the requested answers
print("E1 - One feature model")
print(f"Always-healthy test accuracy: {baseline_accuracy:.1f}%")
print(f"Weight w: {w:.3f}")
print(f"Bias b: {b:.3f}")
print(f"Decision boundary: {boundary:.2f} mm/s")
print(f"Test accuracy: {accuracy:.1f}%")
print(f"Test precision: {precision:.1f}%")
print(f"Test recall: {recall:.1f}%")

E1 - One feature model
Always-healthy test accuracy: 85.0%
Weight w: 2.080
Bias b: -6.835
Decision boundary: 3.29 mm/s
Test accuracy: 89.2%
Test precision: 69.2%
Test recall: 50.0%


In [20]:
print("Columns:", df.columns.tolist())
print("\nSplit labels and counts:")
print(df["split"].value_counts(dropna=False))

print("\nTraining rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Testing rows:", len(test_df))

Columns: ['vib_rms_mm_s', 'kurtosis', 'fault', 'split']

Split labels and counts:
split
train    360
val      120
test     120
Name: count, dtype: int64

Training rows: 360
Validation rows: 0
Testing rows: 120


In [21]:
train_df = df[df["split"] == "train"].copy()
val_df   = df[df["split"] == "val"].copy()
test_df  = df[df["split"] == "test"].copy()

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

Training: (360, 4)
Validation: (120, 4)
Testing: (120, 4)


In [22]:
# E2 - Two feature logistic regression

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

features_E2 = ["vib_rms_mm_s", "kurtosis"]

X_train_E2 = train_df[features_E2]
X_test_E2 = test_df[features_E2]

# StandardScaler is fitted on training rows only
model_E2 = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)

model_E2.fit(X_train_E2, y_train_E)

# Predict at threshold 0.5
prob_E2 = model_E2.predict_proba(X_test_E2)[:, 1]
pred_E2 = (prob_E2 >= 0.5).astype(int)

# Confusion matrix: rows = actual, columns = predicted
tn, fp, fn, tp = confusion_matrix(
    y_test_E, pred_E2, labels=[0, 1]
).ravel()

accuracy_E2 = accuracy_score(y_test_E, pred_E2) * 100
precision_E2 = precision_score(
    y_test_E, pred_E2, zero_division=0
) * 100
recall_E2 = recall_score(
    y_test_E, pred_E2, zero_division=0
) * 100

# Display the assignment's required format
confusion_table = pd.DataFrame(
    [
        [tn, fp],
        [fn, tp]
    ],
    index=["Actually healthy", "Actually faulty"],
    columns=["Predicted healthy", "Predicted faulty"]
)

print("E2 - Test confusion matrix")
display(confusion_table)

print(f"Accuracy: {accuracy_E2:.1f}%")
print(f"Precision: {precision_E2:.1f}%")
print(f"Recall: {recall_E2:.1f}%")

E2 - Test confusion matrix


,Predicted healthy,Predicted faulty
Actually healthy,99,3
Actually faulty,4,14


Accuracy: 94.2%
Precision: 82.4%
Recall: 77.8%


In [23]:
# E3 - Moving the threshold

import numpy as np
import pandas as pd
from sklearn.metrics import recall_score, precision_score, confusion_matrix

# Features must be in the same order used to train model_E2
features = ["vib_rms_mm_s", "kurtosis"]

X_val = val_df[features].copy()
y_val = val_df["fault"].astype(int)

X_test = test_df[features].copy()
y_test = test_df["fault"].astype(int)

# Check data before predicting
if X_val.isnull().any().any() or X_test.isnull().any().any():
    raise ValueError("Validation/test features contain missing values.")

# Validation probabilities: P(fault = 1)
p_val = model_E2.predict_proba(X_val)[:, 1]

# Threshold grid specified in the assignment
thresholds = np.round(np.arange(0.05, 0.951, 0.05), 2)

# Calculate validation recall at each threshold
recalls = []
for threshold in thresholds:
    predictions = (p_val >= threshold).astype(int)
    recalls.append(
        recall_score(y_val, predictions, pos_label=1, zero_division=0)
    )

# Highest validation recall on the grid
highest_val_recall = max(recalls)

# Choose the largest threshold with validation recall >= 0.80
eligible_thresholds = [
    thresholds[i] for i, recall in enumerate(recalls)
    if recall >= 0.80
]

if not eligible_thresholds:
    print("No threshold on this grid achieves validation recall >= 80%.")
else:
    tau = max(eligible_thresholds)

    # Evaluate the chosen threshold on test rows
    p_test = model_E2.predict_proba(X_test)[:, 1]
    pred_test = (p_test >= tau).astype(int)

    test_recall = recall_score(
        y_test, pred_test, pos_label=1, zero_division=0
    )
    test_precision = precision_score(
        y_test, pred_test, pos_label=1, zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test, pred_test, labels=[0, 1]
    ).ravel()

    false_alarm_rate = fp / (fp + tn) if (fp + tn) else 0.0

    # Probability for the specified bearing
    bearing = pd.DataFrame({
        "vib_rms_mm_s": [2.6],
        "kurtosis": [4.2]
    })
    p_fault = model_E2.predict_proba(bearing)[0, 1]

    # Print only the fields requested in E3
    print(f"Chosen threshold tau: {tau:.2f}")
    print(f"Highest validation recall on grid: {highest_val_recall * 100:.1f}%")
    print(f"Test recall at tau: {test_recall * 100:.1f}%")
    print(f"Test precision at tau: {test_precision * 100:.1f}%")
    print(f"Test false-alarm rate: {false_alarm_rate * 100:.1f}%")
    print(f"P(fault) at v=2.6 and kurtosis=4.2: {p_fault:.3f}")

Chosen threshold tau: 0.10
Highest validation recall on grid: 83.3%
Test recall at tau: 83.3%
Test precision at tau: 65.2%
Test false-alarm rate: 7.8%
P(fault) at v=2.6 and kurtosis=4.2: 0.693
